# ReAct, traced step by step

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 16 §16.1–16.2 · type: walkthrough*

**The promise:** build the modern ReAct loop from scratch — the Ch 12 tool loop plus an
iterate-deliberately prompt — and watch it pick each action with everything observed so far in view.


## 🧠 Why this matters

Chapter 12 gave you the *engine*: model proposes an action, your code runs it, the result goes
back into context, repeat. **ReAct** is the first *strategy* you wrap around that engine — it makes
the model alternate **thought → action → observation** until the evidence in hand supports an answer.

The mental model to hold for this whole chapter: a reasoning pattern is a **control-flow design**.
The model supplies judgment *inside* each step; you design the steps. Every pattern in Ch 16 is built
from the same four primitives — **generate** (ask the model), **act** (run a tool), **evaluate**
(check a result), **branch** (decide what's next). Decompose any framework into those four and no
vocabulary will ever mystify you. ReAct is the simplest arrangement: generate → act → observe → loop.


## Objectives & prerequisites

**By the end you can:**
- Implement the modern ReAct loop (tool loop + a deliberate-iteration prompt) from scratch.
- Read a ReAct trace turn-by-turn — thought, action, observation — and see the `max_steps` exit.
- Explain *interleaved thinking*: why you carry reasoning forward across turns, and the per-step cost dial.
- Say when ReAct's adaptivity helps and when its myopia hurts.

**Run first / prereqs:**
- Ch 12 tool loop (`12-02-tool-loop-from-scratch`) — the loop this pattern extends.
- Ch 9 (reasoning / thinking budget) · Ch 11 (model APIs).
- This notebook runs **free and offline** in `MOCK=1` (the default). No API key needed.


## Setup

Imports, `load_dotenv()`, and the `MOCK` switch. **`MOCK=1` is the default** — every model call returns
a canned, realistic response so this notebook is deterministic, free, and offline. Set `COMPANION_MOCK=0`
in your environment (with `ANTHROPIC_API_KEY`) to hit the live API; a full run here is a handful of short calls.


In [ ]:
import os
import json
import random
import textwrap
from dataclasses import dataclass, field

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # python-dotenv is in requirements.txt; absence just means no .env loaded

# MOCK=1 (default) => canned responses, no network, no key. MOCK=0 => live API.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"
MODEL = "claude-opus-4-8"  # the book's default; only used when MOCK=0

random.seed(16)  # determinism for any stochastic mock choices

if MOCK:
    print("MOCK mode: canned responses, offline, free. (set COMPANION_MOCK=0 for live)")
else:
    assert os.getenv("ANTHROPIC_API_KEY"), "MOCK=0 needs ANTHROPIC_API_KEY in your env"
    import anthropic
    client = anthropic.Anthropic()
    print(f"LIVE mode: calling {MODEL}. Expect a few short calls (~cents).")


## A tiny offline tool

ReAct needs something to *act on*. We give it one search-ish tool over three in-memory "documents" so the
whole loop is offline and deterministic. The task is a deliberate **2-hop** question: find who leads a project,
*then* find that person's location — you can't answer it in one lookup.


In [ ]:
DOCS = {
    "project-atlas": "Project Atlas is the agent-platform rewrite. Tech lead: Mara Quinn.",
    "mara-quinn": "Mara Quinn is a staff engineer based in the Lisbon office. Joined 2021.",
    "offices": "Engineering offices: Lisbon (EU), Austin (US), Singapore (APAC).",
}

TOOLS = [
    {
        "name": "search_docs",
        "description": "Look up an internal document by keyword. Returns the matching doc text.",
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    }
]

def run_tool(name: str, args: dict) -> str:
    """Execute a tool call. Pure, offline, deterministic."""
    if name != "search_docs":
        return f"ERROR: unknown tool {name!r}"
    words = args.get("query", "").lower().split()
    # Score each doc by how many query words it matches; return the best (ties -> first).
    best_key, best_score = None, 0
    for key, text in DOCS.items():
        hay = f"{key} {text}".lower()
        score = sum(1 for w in words if w in hay)
        if score > best_score:
            best_key, best_score = key, score
    return DOCS[best_key] if best_key else "No document matched that query."

print(run_tool("search_docs", {"query": "atlas lead"}))


## The mock model: canned tool-use blocks

In `MOCK=1` we stand in for the Messages API with a small state machine that returns the *same shape* the
real SDK does — an object with `.stop_reason`, `.content` (a list of blocks), and `.usage`. It plays the
2-hop script: search for the lead, then search for that person, then answer. This is what lets the loop below
be identical in mock and live mode — only this function changes.


In [ ]:
@dataclass
class Block:
    type: str
    text: str = ""
    id: str = ""
    name: str = ""
    input: dict = field(default_factory=dict)

@dataclass
class Usage:
    input_tokens: int = 0
    output_tokens: int = 0

@dataclass
class Response:
    stop_reason: str
    content: list
    usage: Usage = field(default_factory=Usage)

# The scripted ReAct trajectory for the 2-hop question. Each entry is one model turn.
_MOCK_SCRIPT = [
    Response("tool_use", [
        Block("thinking", text="Two hops: first find who leads Atlas, then where they sit."),
        Block("text", text="I need the Atlas tech lead first."),
        Block("tool_use", id="t1", name="search_docs", input={"query": "project atlas lead"}),
    ], Usage(420, 60)),
    Response("tool_use", [
        Block("thinking", text="Lead is Mara Quinn. Now look up Mara Quinn's location."),
        Block("text", text="Now I need Mara Quinn's office."),
        Block("tool_use", id="t2", name="search_docs", input={"query": "mara quinn office"}),
    ], Usage(540, 55)),
    Response("end_turn", [
        Block("text", text="Project Atlas is led by Mara Quinn, who is based in the Lisbon office."),
    ], Usage(610, 40)),
]

def mock_create(messages, _state={"i": 0}):
    """Return the next scripted turn. Mirrors client.messages.create(...)."""
    i = _state["i"]
    _state["i"] = min(i + 1, len(_MOCK_SCRIPT) - 1)
    return _MOCK_SCRIPT[i]

def reset_mock():
    mock_create.__defaults__[0]["i"] = 0


## Build the ReAct loop

Here is the whole pattern from §16.2 — the Ch 12 tool loop with a *work-step-by-step* system prompt.
The shape is identical whether the model is real or mocked; `call_model` is the only seam.

Note the system prompt: it asks the model to decide what it still needs, use **one** tool to find it,
read the result, and only answer when the evidence supports it. That instruction is what turns a bare
tool loop into ReAct.


In [ ]:
SYSTEM = (
    "You are a research agent. Work step by step: decide what you still "
    "need to know, use one tool to find it, read the result, and repeat. "
    "Only answer when the evidence in hand supports it."
)

def call_model(messages):
    """The one seam between mock and live. Same return shape either way."""
    if MOCK:
        return mock_create(messages)
    return client.messages.create(
        model=MODEL, max_tokens=4096, system=SYSTEM, tools=TOOLS, messages=messages,
    )

def react_agent(task: str, max_steps: int = 12, trace=None):
    messages = [{"role": "user", "content": task}]
    for step in range(max_steps):
        resp = call_model(messages)
        if trace is not None:
            trace.append((step, resp))
        if resp.stop_reason != "tool_use":
            return resp  # final answer
        messages.append({"role": "assistant", "content": resp.content})
        results = [
            {"type": "tool_result", "tool_use_id": b.id,
             "content": run_tool(b.name, b.input)}
            for b in resp.content if b.type == "tool_use"
        ]
        messages.append({"role": "user", "content": results})
    raise RuntimeError("step budget exhausted")  # 16-03 makes this a real budget


### 🔮 Predict

Before you run the next cell: on a **2-hop** question ("Where is the lead of Project Atlas based?"), will the
model answer immediately, or will it call the `search_docs` tool first — and **how many times** before it answers?

Write down your guess (number of tool calls), then run it.


In [ ]:
reset_mock()
trace = []
task = "Where is the lead of Project Atlas based?"
final = react_agent(task, trace=trace)
tool_calls = sum(1 for _, r in trace for b in r.content if b.type == "tool_use")
print(f"tool calls before answering: {tool_calls}")
print("final answer:", next(b.text for b in final.content if b.type == "text"))


**What you just saw.** Two tool calls, then an answer — one hop per fact. The model didn't (and couldn't)
answer in one shot: it had to find the lead *before* it could look up that person's office. That dependency,
discovered at run time rather than planned up front, is exactly what ReAct is good at.


## Read the trace

The whole point of ReAct is that the trace is legible: every move is a thought, an action, or an observation.
Printing it is your first debugging tool — when an agent misbehaves, you read the trace before you touch the prompt.


In [ ]:
def show_trace(task, trace):
    print(f"TASK: {task}\n" + "=" * 60)
    for step, resp in trace:
        for b in resp.content:
            if b.type == "thinking":
                print(f"[{step}] 💭 thought:      {b.text}")
            elif b.type == "text":
                print(f"[{step}] 📝 say:          {b.text}")
            elif b.type == "tool_use":
                print(f"[{step}] 🔧 action:       {b.name}({b.input})")
                print(f"[{step}] 👁️  observation:  {run_tool(b.name, b.input)}")
        print(f"      (stop_reason={resp.stop_reason})")

show_trace(task, trace)


## Interleaved thinking: carry it forward, or throw it away?

Chapter 9 made *thinking* an inference-time knob. The strong move in a ReAct loop is to let thinking happen
**between** tool calls — reason, act, reason about what came back, act again — and to **carry the thinking
forward** across turns instead of discarding it. The model's working-out from step 2 should still be in view
when it reasons at step 3.

Below we contrast the two policies on our trace: the naive loop keeps only actions and results; the interleaved
loop preserves thinking blocks as state you pass back.


In [ ]:
def collect_thinking(trace):
    return [b.text for _, r in trace for b in r.content if b.type == "thinking"]

thoughts = collect_thinking(trace)
print("NAIVE loop  -> thinking discarded each turn; step 3 re-derives from scratch.")
print("INTERLEAVED -> thinking carried forward; step 3 still sees:")
for t in thoughts:
    print("   •", t)
print("\nThe chain of thought is STATE you maintain, not scratch you delete.")


### The per-step thinking dial

The second knob is *how much* to think per step. Thinking tokens cost money and add latency on **every** step
you spend them — a loop that thinks hard before each of 15 tool calls pays the deliberation tax 15 times, most
of it wasted on routine moves. We model the dial as a simple effort level and tally the cost so the trade-off
is visible, not asserted.


In [ ]:
# Toy cost model: thinking tokens by effort level (mirrors Ch 9's budget dial).
THINK_TOKENS = {"none": 0, "low": 200, "high": 1500}

def step_effort(is_hard_fork: bool) -> str:
    # Spend deep thinking only on the hard fork / surprising result.
    return "high" if is_hard_fork else "low"

# Suppose a 15-step run with 2 genuinely hard forks.
steps = [step_effort(is_hard_fork=(i in (3, 9))) for i in range(15)]
smart_cost = sum(THINK_TOKENS[e] for e in steps)
naive_cost = THINK_TOKENS["high"] * 15  # think hard on every step
print(f"think-everywhere : {naive_cost:>6} thinking tokens")
print(f"think-where-it-counts: {smart_cost:>4} thinking tokens  ({smart_cost / naive_cost:.0%} of the cost)")


## ⚠️ Pitfall: a fat tool result drowns the next thought

ReAct quality is gated by what the model **observes**. The loop reasons over each tool result, so a tool that
returns 10k tokens of raw JSON buries the next thought in noise — the single most common reason a ReAct agent
"loses the thread." Shaping tool results (trim, summarize, surface the error) isn't optional polish here; it's
the pattern's **main quality lever**.


In [ ]:
import json as _json

raw = {"office": "Lisbon", **{f"_meta_{i}": "x" * 40 for i in range(60)}}  # realistic API bloat
raw_str = _json.dumps(raw)

def shape_result(blob: dict, keep=("office",)) -> str:
    """Surface only the fields the next thought needs."""
    return _json.dumps({k: blob[k] for k in keep if k in blob})

print(f"raw observation   : {len(raw_str):>5} chars  (drowns the next thought)")
shaped = shape_result(raw)
print(f"shaped observation: {len(shaped):>5} chars  -> {shaped}")


## 🎯 Senior lens

**Spend deep thinking only on the hard fork, not every hop.** ReAct's adaptivity is its strength *and* its tax:
every step is chosen with full context, but every step you let the model deliberate hard, you pay for it in
tokens and latency. A senior dials thinking to the moment — high on the ambiguous fork or the surprising
observation, near-zero on the clear-cut lookup — and shapes every observation before it re-enters the loop.

And know ReAct's edge: it is **myopic**. With no plan, on a long task it can lose the thread, repeat work, and
burn its budget on a dead end before realizing better options existed. That myopia is what the *planning* in
16-02 fixes — and the runaway risk is what the `RunBudget` in 16-03 bounds.


## Recap

- A reasoning pattern is **control-flow design**: generate → act → evaluate → branch. ReAct is the simplest arrangement.
- Modern ReAct is the Ch 12 tool loop **plus a deliberate-iteration prompt** — no regex parsing; tool-use blocks are native.
- The trace (thought / action / observation) is legible by design — read it first when debugging.
- **Interleaved thinking**: carry reasoning forward across turns, and spend the per-step thinking budget only where re-planning changes the outcome.
- **Shape tool results** — a fat observation drowns the next thought; it's ReAct's main quality lever.
- ReAct is adaptive but **myopic**; planning (16-02) and an enforced budget (16-03) are the next layers.


## Exercises

Each task *changes* something and asks you to predict the result first. Solutions live in `solutions/` (Phase 2),
not inline.

1. **Add a 3rd hop.** Extend `DOCS` and the mock script so the question becomes "Which region is the Atlas lead's
   office in?" (lead → office → region). Predict the new tool-call count, then verify.
2. **Force the myopia.** Add a distractor doc that *partially* matches the first query so the model could fetch the
   wrong doc on hop 1. What happens to the trace? How would a plan (16-02) have prevented it?
3. **Dial the thinking.** Mark a *routine* step as `high` effort and a *hard fork* as `none`. Recompute the cost and
   describe the quality you'd expect to lose.
4. **Shape vs. dump.** Feed the raw 10k-char observation into the trace unshaped (mock it) and describe, in one
   sentence, what you'd expect the next thought to do wrong.


In [ ]:
# Exercise 1: extend DOCS + the mock script for a 3-hop question. Predict the tool-call count first.


In [ ]:
# Exercise 2: add a distractor doc and observe how a myopic single-trajectory loop can go wrong.


In [ ]:
# Exercise 3: mis-assign the per-step thinking effort and recompute the cost.


In [ ]:
# Exercise 4: pass the unshaped 10k-char observation through the loop; note what the next thought does.


## Next

- **Next notebook:** [`16-02-plan-execute-reflect-verify.ipynb`](./16-02-plan-execute-reflect-verify.ipynb) —
  layer planning, reflection, and the verification ladder on top of this loop.
- **Then:** [`16-03-runbudget-and-termination-guards.ipynb`](./16-03-runbudget-and-termination-guards.ipynb) —
  bound this loop so it can never run away.
- **Blueprint (the real one):** you built the toy ReAct loop; the production-hardened version lives in
  [`../../../blueprints/agent-loop/`](../../../blueprints/agent-loop/).
- **Book:** see §16.1–16.2.
